In [13]:
import numpy as np
from numpy import random
import itertools

In [27]:
class NK_model(object):
    
    def __init__(self, n, k):
        self.n = n
        self.k = k
        self.start_type = 0    # the initial given start genotype
        self.get_epistic_site()  # get the epistic network among the sites (including the site itself) 
        
        self.allelic_each_site = [[] for _ in range(self.n)]   # will be used to store the episitc combinations at each site
        self.fitness_list = [{} for _ in range(self.n)]   # will be used to store the fitness of epistic combinations at each
                                                          # site
        self.genotype_list = []   # will be used to store the genotypes generated (subset of all the genotypes)
        self.geno_fitness = []   # will be used to store the calculated fitness of genotypes generated
        self.landscape = {}  # will be used to store the generated landscape (subset of the whole landscape)
        
        
        
    @staticmethod
    def generate_start_type(seq_length):
        '''generate a genotype as the start_type
        
        parameters: 
        ----------
            seq_length: int, the length of the wanted genotype sequence
        output: 
        -------
            str that represents the start_type
        '''
        
        a_list = []
        for i in range(seq_length):  # generate a list randomly composed of 0 and 1
            a_list.append(random.randint(0, 2))
            
        start_type_list = []   
        for i in a_list:   # convert the integer elements (0 and 1) in a_list into string elements ('0' and '1') and append 
            start_type_list.append(str(i))   # into start_type_list
            
        start_type = ''.join(start_type_list)  # join the elements in start_type_list into a string
        
        return start_type
    
    
    @staticmethod
    def creat_neighbor(genotype, mutant_num):
        '''creat the mutant_num-mutant neighbors of the given genotype
        
        parameters: 
        ----------
            genotype: str that repesents genotype, eg: "01", "00"
            mutant_num: int, the number of mutant site to creat neighbors
            
        output: 
        -------
            list that store the mutant_num-mutant neighbors of the given genotype
        '''
        
        total_neigh_list= []  # will be used to store the created neighbors of the given genotype
    
        geno_list=list(genotype) # convert the given genotype (str) into a list
    
        indices = list(range(len(geno_list))) # get all the indices for the given genotype
    
        mutant_indices = list(itertools.combinations(indices, mutant_num)) # get all possible combinations of picking 
                                                                    # mutant_num intergers from indices (i.e. mutant sites)
    
        for i in mutant_indices:  # get all its mutant_num -mutant neighbors of the given genotype
            neigh_list = geno_list.copy()
            for j in i:
                if geno_list[j] == '0':
                    neigh_list[j] = '1'
                else:
                    neigh_list[j] = '0'
            
            total_neigh_list.append(''.join(neigh_list))
    
        return total_neigh_list
    
    
    def get_epistic_site(self):
        '''Generate the epistic networks among the sites'''

        self.epistic_site = []
        for i in range(self.n):
            all_site = [x for x in range(self.n) if x != i]  # get the indexes of all sites excluding i
            
            indices = set()   # will be used to store the indexes of epistic sites
            while len(indices) < self.k:
                indices.add(random.randint(0, len(all_site))) # randomly pick the indexes of k elements from all_site

            indices = list(indices)  # convert the set into list
            
            one_site = [i]   # first append the site under study into one_site (a little bit different from 0615 code)
            for j in indices:  # get the epistic sites of site i
                one_site.append(all_site[j])
            
            self.epistic_site.append(one_site) 
           
        
    @staticmethod     
    def get_epistic_combination(genotype, network):
        '''generate the epistic pairs in a given genotype according to the network
        
        parameters: 
        ----------
            genotype: str that repesents a genotype, 
            network: nested list that stores the epistic networks among the sites (self.epistic_site)
            
        output: 
        -------
            list that stores all the epistic combinations in the given genotype
        '''
        
        epistic_combination = []   # will be used to store the epistic combinations in the given genotype
        
        for j in range(len(genotype)):
            one_site = []   
            for i in network[j]:   # get all epistic alleles in one site (store in list one_site)
                one_site.append(genotype[i])
                
            epistic_combination.append(''.join(one_site))  # combine the epistic alleles and store into epistic_combination
                
        return epistic_combination
        
        
    @staticmethod      
    def cal_fitness(epistic_pair, fitness_list):
        '''calculate the fitness of a given genotype according to the fitness_list (here the epistic_pair is generate by
        the genotype, i.e. the epistic_combination in get_epistic_combination(genotype, network) method)
        
        parameters: 
        ----------
            epistic_pair: list that stores the epistic combinations in a genotype  
            fitness_list: list that stores the dictionary which indicates the fitness of each epistic combinations 
                          in each site
            
        output: 
        -------
            float, the calculated fitness of a genotype (the genotype which corresponded to epistic_pair)
        '''
        
        one_fitness = []    
        
        for i in range(len(epistic_pair)):  # fitness_list[i] is the fitness dictionary that corresponds to site i.  
            one_fitness.append(fitness_list[i][epistic_pair[i]])  # epistic_pair[i] is the epistic combination in site i (the
                                                                # key in the dictionary)
        i_fitness = sum(one_fitness)/len(one_fitness)  # calculate the fitness
            
        return i_fitness
          
        
    @staticmethod
    def whether_fixed(fitness_1, fitness_2):
        
        s = (fitness_2/fitness_1)-1
        fix_pro = 2*s
        
        random_no = random.random(1)[0]
        
        if fix_pro > random_no:
            return True
        else:
            return False
    
    
    
    def adaptive_walk(self):
        '''Simulate one step  of adaptive walk along the landscape'''
        
        if self.start_type==0:  # self.start_type==0 (the initial given value), means initial step, randomly generates a 
                                # genotype as start_type (length is self.n)
            self.start_type = self.generate_start_type(self.n)
        else:  # start!='', not initial step, just pass
            pass
        
        neighbor_list = [self.start_type] # this list will be used to store the self.start_type and all its neighbors
                                        # (next step will select from the neighbors of self.start_type)
        
        neighbor_list += self.creat_neighbor(self.start_type, 1) # get all 1-mutant neighbors of start_type
                
        all_epistic_pair = [] # this list will be used to store all epistic combinations for genotypes in neighbor_list
        
        for i in neighbor_list:  # i is genotype in neighbor_list, self.epistic_site is network among the site
            all_epistic_pair.append(self.get_epistic_combination(i, self.epistic_site))
        
        
        allelic_set = []  # list that will be used to store all the epistic combinations in site 0, 1, 2, ..., self.n-1 
                        # (use set to exclude the repeat ones) for genotypes in neighbor_list
        
        for j in range(self.n): 
            pos_allelic_pair = [] # will be used to store all epistic combinations in site j for genotypes in neighbor_list
            for s in all_epistic_pair:
                pos_allelic_pair.append(s[j])   
            
            pos_allelic_set = list(set(pos_allelic_pair))  # convert the pos_allelic_pair to set then convert back to list to
                                                    # get the non-repeated epistic combinations in site j
                
            allelic_set.append(pos_allelic_set)  # append the non-repeated epistic combinations in site j into allelic_set to
                                        # get epistic combinations in all sites
            
        for s in range(self.n): # the length of allelic_set is self.n
            for i in allelic_set[s]: # i is the allelic combinations in site s, self.allelic_each_site is used to store the 
                                    #  episitc combinations at each site, self.allelic_each_site[s] is episitc combinations 
                                    # at site s that has already been generated
                if i not in self.allelic_each_site[s]: # if i is not in self.allelic_each_site[s], means epistic combination i
                                 # has not been generated, just append i into self.allelic_each_site[s] and generate a fitness
                                 # for i and store into self.fitness_list[s](i will be the key)
                    self.allelic_each_site[s].append(i)
                    self.fitness_list[s][i] = random.random(1)[0]
            
        w_fitness = []   # will be used to store the calculated fitness of each genotype in neighbor_list
        
        for i in neighbor_list:
            if i not in self.genotype_list: # i not in self.genotype_list, means the fitness of genotype i has not been
                                            # calculated--> need to be calculated this time
                index_i = neighbor_list.index(i) # first get the index of i (this index is the same as the epistic 
                                                # combinations for genotype i in all_epistic_pair)
                i_fitness = self.cal_fitness(all_epistic_pair[index_i], self.fitness_list)
                w_fitness.append(i_fitness)
            else:  # otherwise it means that the fitness of genotype i has been calculated, just find it from the landscape
                w_fitness.append(self.landscape[i])
            
        neigh_fitness = dict(zip(neighbor_list[1:], w_fitness[1:]))  # creat a dictionary to store the genotypes and fitness of 
                                                            # the 1-mutant neighbors of self.start_type (start_type is not 
                                                            # included as it won't be selected for self.next_step) 
                                                            # key: genotype; value: fitness
    
        self.start_type_fitness = w_fitness[0]  # get the fitness of self.start_type
        
        possible_neigh = {k : v for k, v in neigh_fitness.items() if v >= self.start_type_fitness}  # get all possible   
                                                                    # neighbors for self.next_step (only if the fitness >= 
                                                                    # self.start_type fitness can be possible neighbors 
                                                                    # for self.next_step) 
        
        if possible_neigh:   # if possible_neigh is not empty, it means that possible genotypes for self.next_step exist;
                             # select the genotype for self.next_step according to their fitness
                
            pop_gens = list(possible_neigh.keys())   # get all the possible genotypes of self.next_step
            possible_next_type = pop_gens[random.randint(0, len(pop_gens))]  # find the possible_next_type and 
            possible_next_type_fit = possible_neigh[possible_next_type] # possible_next_type_fitness
            
            if self.whether_fixed(self.start_type_fitness, possible_next_type_fit): # check if the next_type can be fixed
                self.next_type = possible_next_type         # if can be fixed, then updated the self.next_type and its fitness
                self.next_type_fitness = possible_next_type_fit
            else:
                self.next_type = self.start_type
                self.next_type_fitness = self.start_type_fitness
                                                                                
        else: # if possible_neigh is empty, it means that there if no option for self.next_step-->cannot evolve
              #-->reach local optima
            self.next_type = ''
            self.next_type_fitness = 0   
           
    
        self.genotype_list.extend(neighbor_list)  # updating the self.genotype_list by storing all the genotype in neighbor_list
                                                # into self.genotype_list
            
        self.genotype_list = list(set(self.genotype_list))  # exclude the repeating genotypes in self.genotype_list
        
        landscape_subset = dict(zip(neighbor_list, w_fitness)) # the subset of landscape generated in this random_walk step
        
        self.landscape.update(landscape_subset)  # update self.landscape by the subset of landscape generated in this random
                                                 # _walk step
                
        
    def repeat_adaptive_walk(self):
        '''Simulate the random walk in the landscape'''
        
        self.walk_list = []  # this list will be used to store the steps of random walk
        
        fit = self.adaptive_walk()  # run the random_walk method
        
        self.walk_list.append([self.start_type, self.start_type_fitness])  # the initial step
        
        while self.next_type and self.next_type_fitness:  # this means that while next_step!='' and fitness[next_step]!=0.0,  
                                                         #not reach local optima
            self.walk_list.append([self.next_type, self.next_type_fitness]) # the next_step
            self.start_type = self.next_type  # update the self.start_type and self.start_type_fitness
            self.start_type_fitness = self.next_type_fitness
            fit = self.adaptive_walk() # run the random_walk method with the updated self.start_type 
        
        self.walk_genotype = list(set([i[0] for i in self.walk_list])) 
        self.new_walk_list = [[j, self.landscape[j]] for j in self.walk_genotype]
        self.new_walk_list.sort(key=lambda x: x[1])
        self.walk_genotype = [i[0] for i in self.new_walk_list]
 
        if len(self.new_walk_list) > 6:
            self.walk_genotype = self.walk_genotype[:6]
            self.new_walk_list = self.new_walk_list[:6]
            
    
#     def multiple_adaptive_walk(self, times):
#         '''Allow multiple random walks in a landscape'''
        
#         self.multiple_walk_list = []  # will be used to store the walk
        
#         for i in range(times):
#             self.repeat_adaptive_walk()   
#             self.multiple_walk_list.append(self.walk_list)
#             self.start_type = self.generate_start_type(self.n)
    
    
    def search_different_allele(self):
        self.repeat_adaptive_walk()
        
        walk_geno_list = []
        for i in self.walk_genotype:
            walk_geno_list.append(list(i))
            
        self.total_allele_list = []
        for i in range(self.n):
            allele_list = []
            for j in walk_geno_list:
                allele_list.append(j[i])
                
            allele_list = list(set(allele_list))
            self.total_allele_list.append(allele_list)
                
        
        self.shared_allele_indice = []
        self.diff_allele_indice = []
        
        for i in range(self.n):
            if len(self.total_allele_list[i])>1:
                self.diff_allele_indice.append(i)
            else:
                self.shared_allele_indice.append(i)
                
        print('DIFF INDICE', self.diff_allele_indice)
#         print('SHARED', self.shared_allele_indice)
        
        self.shared_allele = ''
        
        for i in self.shared_allele_indice:
            self.shared_allele+=self.walk_genotype[0][i]
            
        print('SHARED ALLE', self.shared_allele)
        
        self.landscape_subset = {}
        for i in self.walk_genotype:
            s = ''
            for j in self.diff_allele_indice:
                s+=i[j]

            self.landscape_subset[s] = self.landscape[i]
            
            
      
    def creat_landscape_subset(self):
        
        self.search_different_allele()
              
        all_possible_combination = [''.join(i) for i in itertools.product('01', repeat = len(self.diff_allele_indice))]

        already_find = list(self.landscape_subset.keys())
        need_to_find_com = []

        for i in all_possible_combination:
             if i not in already_find:
                need_to_find_com.append(i)

        need_to_find_com_list = []
        for i in need_to_find_com:
            need_to_find_com_list.append(list(i))


        self.shared_allele_list = list(self.shared_allele)

        orginal_genotype = []
        for i in need_to_find_com_list:
            for j in self.shared_allele_indice:
                index_j = self.shared_allele_indice.index(j)
                i.insert(j, self.shared_allele_list[index_j])
            orginal_genotype.append(''.join(i))

    #         print('JOIN', orginal_genotype, len(orginal_genotype))

        orginal_genotype_fitness = []

        all_geno = list(self.landscape.keys())
    #         print('ALL', all_geno)

        for i in orginal_genotype:
            if i in all_geno:
                orginal_genotype_fitness.append(self.landscape[i])
            else:
    #                 print('NOT IN All')
                allelic_comb_i = self.get_epistic_combination(i, self.epistic_site)
    #                 print('AC', allelic_comb_i)
                for j in range(self.n):
                    if allelic_comb_i[j] not in self.allelic_each_site[j]:
                        self.allelic_each_site[j].append(allelic_comb_i[j])
                        self.fitness_list[j][allelic_comb_i[j]] = random.random(1)[0]

                fitness_i = self.cal_fitness(allelic_comb_i, self.fitness_list)

                orginal_genotype_fitness.append(fitness_i)
                self.landscape[i] = fitness_i

    #         print('FIT', orginal_genotype_fitness)   
        updated_landscape_subset = dict(zip(need_to_find_com, orginal_genotype_fitness))

        self.landscape_subset.update(updated_landscape_subset)
    
    
   

In [32]:
total_subset = []

while len(total_subset)<10:
    a = NK_model(20, 7)
#     a.repeat_adaptive_walk()
    a.creat_landscape_subset()
    
    if len(a.diff_allele_indice) ==5:
        total_subset.append(a.landscape_subset)
    
print('TOTAL SUBSET', total_subset, len(total_subset))

DIFF INDICE [6, 14]
SHARED ALLE 110001100100011010
DIFF INDICE [0, 1, 3, 7, 8]
SHARED ALLE 101101010010000
DIFF INDICE [5, 7, 10, 14, 18]
SHARED ALLE 101100111101010
DIFF INDICE [0, 7, 12, 14, 15]
SHARED ALLE 001011100011010
DIFF INDICE [0, 3, 6, 12]
SHARED ALLE 1110000000100000
DIFF INDICE [0, 1, 4, 6, 18]
SHARED ALLE 000100101100100
DIFF INDICE [0, 1, 8, 9, 14]
SHARED ALLE 010101010100011
DIFF INDICE [4]
SHARED ALLE 0001010001101111110
DIFF INDICE [0, 2, 14]
SHARED ALLE 01011110010011100
DIFF INDICE [0, 11, 16]
SHARED ALLE 00100011110011111
DIFF INDICE [0, 12, 17]
SHARED ALLE 11100011010100000
DIFF INDICE [7, 8, 9, 12, 14]
SHARED ALLE 101111101001111
DIFF INDICE [0, 4, 5, 10, 16]
SHARED ALLE 111111101100001
DIFF INDICE [2, 6, 8, 11, 13]
SHARED ALLE 100100100110111
DIFF INDICE [3, 4, 10, 18, 19]
SHARED ALLE 000001010101101
DIFF INDICE [0, 6, 8, 15, 17]
SHARED ALLE 110100010110010
TOTAL SUBSET [{'11111': 0.43768251884545262, '11010': 0.53379811388818199, '10100': 0.57727100493201589, '

In [21]:
a = NK_model(20,7)
# print(a.epistic_site)
# a.random_walk()
a.repeat_adaptive_walk()
# # a.multiple_random_walk(3)
# # print('1', a.walk_list)
# # print('2', a.landscape)
# # print('LEN', len(list(a.landscape.keys())))
# # print('3', a.walk_genotype)
print('4', a.new_walk_list, len(a.new_walk_list))


# a.search_different_allele()

a.creat_landscape_subset()
print('5', a.landscape_subset, len(list(a.landscape_subset.keys())))



4 [['11111010100100101010', 0.49201109431667633], ['11111010100100001010', 0.55066332324792733], ['11111010100100001110', 0.59473599427132717], ['11111011100100001110', 0.61839834902319901], ['11111011100000001110', 0.63320230672056788], ['11011011100000001110', 0.63893587925109008]] 6
DIFF INDICE [2, 7, 11, 14, 17]
SHARED ALLE 111101100000110
5 {'00001': 0.54912240681872682, '11010': 0.5688887922232494, '10100': 0.55066332324792733, '01011': 0.59625431619136493, '00011': 0.55665750580613027, '10010': 0.55243406691687857, '11110': 0.53766818128577332, '11001': 0.63320230672056788, '10111': 0.55048607104460667, '01111': 0.55737964239971949, '11111': 0.56803330104469718, '10101': 0.59473599427132717, '00101': 0.54011996614978219, '01100': 0.60580388230766968, '00111': 0.489971948862995, '11011': 0.57473438241855523, '00110': 0.43149697213506466, '10011': 0.58499803556995444, '11000': 0.65693202147675922, '11100': 0.61055944689258057, '01000': 0.66266559400728142, '00000': 0.5538655640751